# آموزش واحد MoE: متن ChatGPT + logits معلم Qwen

این نوت‌بوک فقط یک مسیر آموزش دارد: **پاسخ‌های متنی ChatGPT + تقطیر sparse logits مدل معلم Qwen** در همان حلقهٔ آموزش مدل MoE. مسیرهای متن‌تنها و logits‌تنها اجرا نمی‌شوند.

آرشیو متن روی Google Drive نگهداری می‌شود و در GitHub commit نمی‌شود. ابتدا logits معلم برای همان توکن‌های پاسخ ذخیره می‌شود؛ سپس مدل پایه با ۲۴ checkpoint اولیهٔ MoE patch می‌شود و فقط FFNهای MoE/router آموزش می‌بینند. Backbone اصلی ثابت است.

**هدف معماری فعلی:** حدود ۱٫۰۸۶B پارامتر فعال؛ این نوت‌بوک معماری را تغییر نمی‌دهد. اجرای واقعی نیازمند GPU و فضای Drive کافی است. کد به‌صورت خودکار تست کیفیت یا مقایسهٔ روش‌های جایگزین اجرا نمی‌کند.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess, sys, json

REPO = Path('/content/Qwen-MoE-0.4B-')
WORK = Path('/content/drive/MyDrive/qwen-moe-chatgpt-logits')
INPUT_JSONL = WORK / 'chatgpt_prompt_response_archive.jsonl'
LOGITS_OUT = WORK / 'qwen_logits_output'
INITIAL_CHECKPOINTS = Path('/content/drive/MyDrive/qwen-moe-full-build/checkpoints')
TRAIN_OUT = WORK / 'combined_moe_training'
MODEL_ID = 'Qwen/Qwen3.5-2B'
TOP_K = 64
TEMPERATURE = 1.0
MAX_SEQ_LENGTH = 256

WORK.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'safetensors'], check=True)

required = {
    'ChatGPT text archive': INPUT_JSONL,
    'initial 24-layer MoE checkpoints': INITIAL_CHECKPOINTS,
}
for label, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f'{label} not found: {path}')

if not __import__('torch').cuda.is_available():
    raise RuntimeError('CUDA GPU is not available. Select a Colab GPU runtime before running training.')

print('Text archive:', INPUT_JSONL)
print('Teacher-logit output:', LOGITS_OUT)
print('Initial MoE checkpoints:', INITIAL_CHECKPOINTS)
print('Combined training output:', TRAIN_OUT)
print('Model:', MODEL_ID)
print('Pipeline: ChatGPT response CE + Qwen sparse-logit KD only')

## قالب دقیق آرشیو متن

فایل `chatgpt_prompt_response_archive.jsonl` باید هر خط یک JSON object مستقل داشته باشد و حداقل این فیلدها را داشته باشد:

- `sample_id`: شناسهٔ یکتا و پایدار
- `prompt`: پرسش/دستور ورودی
- `chatgpt_response`: پاسخ کامل، باکیفیت و آمادهٔ آموزش

نمونهٔ زیر فقط قالب را نشان می‌دهد و خودکار به فایل اضافه نمی‌شود. متن واقعی را از ایجنت تولیدکننده وارد کن.

In [ ]:
example = {
    "sample_id": "fa-reasoning-000001",
    "prompt": "چرا تقسیم بر صفر در حساب معمول تعریف نشده است؟",
    "chatgpt_response": "در تقسیم a بر b، عددی می‌خواهیم که ضرب آن در b برابر a شود. وقتی b صفر باشد، برای a غیرصفر چنین عددی وجود ندارد؛ و برای a برابر صفر نیز جواب یکتا نیست."
}
print(json.dumps(example, ensure_ascii=False, indent=2))
print('Example only; it is not appended to the training archive.')

## مرحلهٔ ۱ — استخراج logits معلم

اسکریپت توالی دقیق prompt+answer، توکن‌های هدف، logits برتر Qwen و log-normalizer کل واژگان را ذخیره می‌کند. log-normalizer برای حفظ جرم احتمالاتی خارج از top-k استفاده می‌شود. این KD یک تقریب با tail تجمیع‌شده است، نه KL دقیق روی تمام واژگان.

خروجی‌ها روی Drive ذخیره می‌شوند و اجرای دوباره فایل‌های tensor موجود را رد می‌کند. آرشیو متن اصلی تغییر نمی‌کند.

In [ ]:
subprocess.run([
    sys.executable, str(REPO / 'scripts/capture_qwen_logits_for_chatgpt_data.py'),
    '--input-jsonl', str(INPUT_JSONL),
    '--output-dir', str(LOGITS_OUT),
    '--model', MODEL_ID,
    '--top-k', str(TOP_K),
    '--temperature', str(TEMPERATURE),
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--device', 'auto',
], check=True)

INDEX_JSONL = LOGITS_OUT / 'qwen_logits_index.jsonl'
MANIFEST_JSON = LOGITS_OUT / 'qwen_logits_manifest.json'
print('Index:', INDEX_JSONL)
print('Manifest:', MANIFEST_JSON)
print(json.loads(MANIFEST_JSON.read_text(encoding='utf-8'))) 

## مرحلهٔ ۲ — آموزش ترکیبی واحد

آموزش از ۲۴ checkpoint اولیهٔ MoE شروع می‌شود. مدل پایه با همان revision ثابت بارگذاری می‌شود، FFNهای dense با checkpointهای MoE جایگزین می‌شوند، وزن‌های backbone فریز می‌شوند و فقط expertها و routerها آموزش می‌بینند.

تابع هدف واحد است:

`total_loss = text_cross_entropy + 0.5 × qwen_sparse_KD`

Loss تقطیر از logits ذخیره‌شده استفاده می‌کند و جرم توکن‌های خارج از top-k را به‌صورت یک bucket تجمیع می‌کند. این نوت‌بوک هیچ اجرای text-only یا logits-only ندارد. تنظیمات پیش‌فرض یک epoch است؛ خروجی و وضعیت optimizer روی Drive ذخیره می‌شوند.

In [ ]:
TRAIN_OUT.mkdir(parents=True, exist_ok=True)
train_cmd = [
    sys.executable, str(REPO / 'scripts/train_moe_chatgpt_qwen_kd.py'),
    '--index-jsonl', str(INDEX_JSONL),
    '--checkpoint-dir', str(INITIAL_CHECKPOINTS),
    '--output-dir', str(TRAIN_OUT),
    '--model', MODEL_ID,
    '--num-experts', '32',
    '--top-k', '2',
    '--epochs', '1',
    '--learning-rate', '1e-5',
    '--kd-weight', '0.5',
    '--temperature', str(TEMPERATURE),
    '--save-every', '50',
    '--device', 'auto',
]
# If this run was interrupted, continue from the saved MoE checkpoint/optimizer state.
if (TRAIN_OUT / 'training_state.pt').is_file() and (TRAIN_OUT / 'checkpoints').is_dir():
    train_cmd.append('--resume')
print('Running the single combined training route.')
subprocess.run(train_cmd, check=True)

## مرحلهٔ ۳ — اعتبارسنجی ساختاری checkpoint نهایی

این مرحله فقط کامل‌بودن و سازگاری ساختاری ۲۴ checkpoint را بررسی می‌کند؛ ادعای بهبود کیفیت یا آمادگی استقرار نیست. برای ارزیابی کیفیت باید مجموعهٔ ارزیابی مستقل و از قبل کنارگذاشته‌شده استفاده شود؛ دادهٔ آموزش نباید به‌عنوان ارزیابی گزارش شود.

In [ ]:
VALIDATION_JSON = TRAIN_OUT / 'checkpoint_validation.json'
subprocess.run([
    sys.executable, str(REPO / 'scripts/validate_checkpoint_set.py'),
    '--checkpoint-dir', str(TRAIN_OUT / 'checkpoints'),
    '--layers', '24',
    '--num-experts', '32',
    '--top-k', '2',
    '--output', str(VALIDATION_JSON),
], check=True)
print(VALIDATION_JSON.read_text(encoding='utf-8'))
print('Training summary:')
print((TRAIN_OUT / 'training_summary.json').read_text(encoding='utf-8'))

## فایل‌های خروجی روی Drive

- `qwen_logits_output/qwen_logits_index.jsonl` و پوشهٔ `qwen_topk_logits/`: سیگنال عددی معلم
- `combined_moe_training/checkpoints/`: checkpointهای نهایی آموزش‌شده
- `combined_moe_training/training_metrics.jsonl`: Loss و گرادیان هر مرحله
- `combined_moe_training/training_state.pt`: وضعیت optimizer برای ادامهٔ آموزش
- `combined_moe_training/training_summary.json`: خلاصهٔ اجرا

محدودیت‌ها: این نوت‌بوک و کد باید روی GPU واقعی اجرا شوند تا اجرای موفق تأیید شود. تقطیر top-k با tail تجمیع‌شده تقریب KL است. این روش به‌تنهایی تضمین کیفیت پاسخ یا اجرای سریع روی Android نیست.